In [ ]:
######### Estimate the combined splicing effect
# link: https://www.alphagenomedocs.com/colabs/splicing_variant_scoring.html
# formula: alphagenome_splicing = max(splice_sites) + max(splice_site_usage) + max(splice_junctions) / 5.0

In [6]:
from alphagenome import colab_utils
from alphagenome.data import genome
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
import pandas as pd

In [7]:
# Initialize AlphaGenome DNA model
dna_model = dna_client.create(
    api_key="YOUR_API_KEY_HERE"
)

In [8]:
splicing_scorers = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITES'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITE_USAGE'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_JUNCTIONS'],
]

for scorer in splicing_scorers:
  print(f'{scorer.name} (signed={scorer.is_signed})')

GeneMaskSplicingScorer(requested_output=SPLICE_SITES, width=None) (signed=False)
GeneMaskSplicingScorer(requested_output=SPLICE_SITE_USAGE, width=None) (signed=False)
SpliceJunctionScorer() (signed=False)


In [9]:
# Variant id: chr9:21968347:T>C
variant = genome.Variant(
    chromosome='chr9',
    position=21968347,
    reference_bases='T',
    alternate_bases='C',
)

# Create a 1MB interval centered on the variant.
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

# Score the variant using the splicing scorers.
scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=splicing_scorers,
    organism=dna_client.Organism.HOMO_SAPIENS,
)

# View the tidy scores.
df_scores = variant_scorers.tidy_scores([scores])
df_scores

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,...,track_strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,gtex_tissue,data_source,raw_score,quantile_score
0,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,None,None,SPLICE_SITES,GeneMaskSplicingScorer(requested_output=SPLICE...,...,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.808411,0.999980
1,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,None,None,SPLICE_SITES,GeneMaskSplicingScorer(requested_output=SPLICE...,...,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.136719,0.994071
2,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,-,polyA plus RNA-seq,CL:0000047,neuronal stem cell,in_vitro_differentiated_cells,embryonic,,encode,0.062500,0.997285
3,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,-,total RNA-seq,CL:0000062,osteoblast,primary_cell,adult,,encode,0.078125,0.998608
4,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,-,polyA plus RNA-seq,CL:0000084,T-cell,primary_cell,adult,,encode,0.078125,0.998815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,21968242,21968347,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,polyA plus RNA-seq,UBERON:0018116,right renal pelvis,tissue,embryonic,,encode,1.614258,0.999759
732,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,21968242,21968347,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,polyA plus RNA-seq,UBERON:0018117,left renal cortex interstitium,tissue,embryonic,,encode,1.603516,0.999759
733,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,21968242,21968347,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,polyA plus RNA-seq,UBERON:0018118,right renal cortex interstitium,tissue,embryonic,,encode,1.415039,0.999696
734,chr9:21968347:T>C,chr9:21444059-22492635:.,ENSG00000147889,CDKN2A,protein_coding,-,21968242,21968347,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,polyA plus RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,Skin_Not_Sun_Exposed_Suprapubic,gtex,2.144531,0.999892


In [10]:
def compute_merged_splicing_score(
    df: pd.DataFrame,
) -> pd.DataFrame:
  """Computes the merged splicing score from tidy variant scores.

  For each variant, takes the max raw_score across genes and tracks for each
  splicing scorer, then combines them as:
    merged_splicing = max(splice_sites) + max(splice_site_usage)
                      + max(splice_junctions) / 5

  Args:
    df: Tidy scores DataFrame from variant_scorers.tidy_scores().

  Returns:
    DataFrame with one row per variant and columns for each splicing scorer's
    max score plus the merged splicing score.
  """
  # Get the max score per variant per output type.
  df['variant_id'] = df['variant_id'].map(str)
  max_scores = (
      df.groupby(['variant_id', 'output_type'])['raw_score']
      .max()
      .reset_index()
      .pivot(index='variant_id', columns='output_type', values='raw_score')
      .fillna(0.0)
  )

  # Compute the merged splicing score with the appropriate weights.
  max_scores['alphagenome_splicing'] = (
      max_scores.get('SPLICE_SITES', 0.0)
      + max_scores.get('SPLICE_SITE_USAGE', 0.0)
      + max_scores.get('SPLICE_JUNCTIONS', 0.0) / 5.0
  )

  return max_scores.reset_index()


merged_scores = compute_merged_splicing_score(df_scores)
merged_scores

output_type,variant_id,SPLICE_JUNCTIONS,SPLICE_SITES,SPLICE_SITE_USAGE,alphagenome_splicing
0,chr9:21968347:T>C,3.380859,0.808411,0.151718,1.636301
